# GTSRB frequency-domain adversarial robustness: FreqDebias-AT vs DAT
Colab runner (CLAUDE.md §10). **Runtime → Change runtime type → GPU** before starting.

Run the *one-time setup* cells top to bottom, then one cell per run (§10.2). After a disconnect: reconnect, re-run setup cells 2–6, and re-run the **same** training cell; `--resume` continues from `last.pt`.

## 10.1 One-time setup

In [ ]:
# 2. GPU check
import subprocess, sys
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout[:1500])
if out.returncode != 0:
    raise SystemExit('\n\n*** NO GPU: Runtime -> Change runtime type -> GPU, then restart this notebook ***\n')

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Paths
import os
OUT  = '/content/drive/MyDrive/gtsrb_freq_robust'   # all results, checkpoints, caches (resumable)
DATA = '/content/data'                               # GTSRB download (session-local)
REPO = '/content/repo'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
os.environ['OUT'], os.environ['DATA'], os.environ['REPO'] = OUT, DATA, REPO
print(OUT)

In [ ]:
# 5. Clone / pull the code and install the extras (torch/torchvision = Colab's preinstalled versions)
GITHUB_REPO = 'https://github.com/NeuralNomadic/DAT-and-FreqDebias.git'   # <-- set this
import os, subprocess
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')          # Colab Secrets, only needed for a private repo
except Exception:
    pass
url = GITHUB_REPO if not token else GITHUB_REPO.replace('https://', f'https://{token}@')
if os.path.isdir(os.path.join(REPO, '.git')):
    print(subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], capture_output=True, text=True).stdout)
else:
    subprocess.run(['git', 'clone', url, REPO], check=True)
%cd $REPO
!pip install -q -r requirements.txt
!git log -1 --oneline

In [ ]:
# 6. Download GTSRB (skips if already present this session)
import torchvision, os
if not os.path.isdir(os.path.join(DATA, 'gtsrb', 'GTSRB')):
    torchvision.datasets.GTSRB(DATA, split='train', download=True)
    torchvision.datasets.GTSRB(DATA, split='test', download=True)
print('GTSRB ready')

In [ ]:
# 7. (first time only) split by track + 32x32 caches + stats.json + class_names.json + aa_subset.txt
!python -m src.make_split --data $DATA --out $OUT

In [ ]:
# Record the exact environment (after the first successful run)
!mkdir -p $OUT/env && pip freeze > $OUT/env/requirements.lock.txt && head -3 $OUT/env/requirements.lock.txt

In [ ]:
# Optional sanity: all tests on CPU (~5 min). Required to pass before the first GPU run (CLAUDE.md §11).
!python -m pytest tests -q -x

## 10.2 Runs — Tier 1 (minimum thesis result): E0, E3, E5, E6, seed 0
Each cell trains (resumable) and then evaluates. Progress without stopping: open `runs/<EXP>/seed<k>/train_log.csv` on Drive.

In [ ]:
!python -m src.train    --config configs/E0_standard.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E0_standard --seed 0 --data $DATA --out $OUT

In [ ]:
# Milestone 2: E3 (control). Report sec/epoch from train_log.csv to plan the tiers.
!python -m src.train    --config configs/E3_base.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E3_base --seed 0 --data $DATA --out $OUT

In [ ]:
# Milestone 3: Fo-Mixup masks (needs E0 best.pt) + fomixup_grid.png -> inspect OUT/figures/fomixup_grid.png
!python -m src.precompute_fomixup --data $DATA --out $OUT
from IPython.display import Image, display
display(Image(f'{OUT}/figures/fomixup_grid.png'))

In [ ]:
!python -m src.train    --config configs/E5_fd_at.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E5_fd_at --seed 0 --data $DATA --out $OUT

In [ ]:
# Milestone 4: DAT. dat_grid.png first (random generator), again after training (learned generator).
!python -m src.make_grids --data $DATA --out $OUT --which dat
!python -m src.train    --config configs/E6_dat.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E6_dat --seed 0 --data $DATA --out $OUT
!python -m src.make_grids --data $DATA --out $OUT --which dat

In [ ]:
# Milestone 5: tables and figures for everything evaluated so far
!python -m src.aggregate --out $OUT

## Tier 2: E1, E2, E4 (seed 0)

In [ ]:
!python -m src.train    --config configs/E1_pgdat.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E1_pgdat --seed 0 --data $DATA --out $OUT

In [ ]:
!python -m src.train    --config configs/E2_trades.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E2_trades --seed 0 --data $DATA --out $OUT

In [ ]:
!python -m src.train    --config configs/E4_fd_clean.yaml --seed 0 --data $DATA --out $OUT --resume
!python -m src.evaluate --exp E4_fd_clean --seed 0 --data $DATA --out $OUT

## Tier 3: seeds 1, 2 of E3, E5, E6

In [ ]:
for exp in ['E3_base', 'E5_fd_at', 'E6_dat']:
    for seed in [1, 2]:
        !python -m src.train    --config configs/{exp}.yaml --seed {seed} --data $DATA --out $OUT --resume
        !python -m src.evaluate --exp {exp} --seed {seed} --data $DATA --out $OUT

## Tier 4: ablations A1–A6 (seed 0)

In [ ]:
for exp in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']:
    !python -m src.train    --config configs/{exp}.yaml --seed 0 --data $DATA --out $OUT --resume
    !python -m src.evaluate --exp {exp} --seed 0 --data $DATA --out $OUT

## Tier 5: seeds 1, 2 of E1, E2

In [ ]:
for exp in ['E1_pgdat', 'E2_trades']:
    for seed in [1, 2]:
        !python -m src.train    --config configs/{exp}.yaml --seed {seed} --data $DATA --out $OUT --resume
        !python -m src.evaluate --exp {exp} --seed {seed} --data $DATA --out $OUT

## Aggregate (re-run any time)

In [ ]:
!python -m src.aggregate --out $OUT
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/results/table_main.md').read()))